In [1]:
# Cell 1: Environment, Paths & NVIDIA Dynamic Library Pre-flight Diagnostics
import sys
import os
import ctypes
import site
from pathlib import Path

# 1. Ensure project root is in sys.path
root_dir = Path.cwd().resolve()
if root_dir.name == "terrain_diffusion":
    root_dir = root_dir.parent
elif root_dir.name == "working":
    root_dir = root_dir.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

print("=" * 65)
print("1. ENVIRONMENT, PATHS & NVIDIA LIBRARY PRE-FLIGHT SETUP")
print("=" * 65)
print(f"Python Executable   : {sys.executable}")
print(f"Working Directory   : {os.getcwd()}")
print(f"Project Root        : {root_dir}")
print(f"CONDA_PREFIX        : {os.environ.get('CONDA_PREFIX', '(not set)')}")
print(f"LD_LIBRARY_PATH     : {os.environ.get('LD_LIBRARY_PATH', '(not set)')}")
print(f"CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES', '(not set)')}")

# 2. Pre-flight check and load for NVIDIA shared libraries (cuDNN / cuBLAS)
# In cuDNN 9, dynamic loading of cudnnGetVersion from sibling libraries requires
# either LD_LIBRARY_PATH or pre-loading into the global symbol table.
loaded_nvidia_libs = []
site_dirs = site.getsitepackages()
if hasattr(site, 'getusersitepackages'):
    site_dirs.append(site.getusersitepackages())

print()
print("--- Scanning & Loading NVIDIA Shared Libraries from Site-Packages ---")
for s_dir in site_dirs:
    for pkg in ["cudnn", "cublas", "cuda_runtime"]:
        lib_dir = os.path.join(s_dir, "nvidia", pkg, "lib")
        if os.path.isdir(lib_dir):
            for fname in sorted(os.listdir(lib_dir)):
                if fname.endswith(".so.9") or fname.endswith(".so.12") or fname.endswith(".so.13"):
                    full_path = os.path.join(lib_dir, fname)
                    try:
                        ctypes.CDLL(full_path, mode=ctypes.RTLD_GLOBAL)
                        loaded_nvidia_libs.append(fname)
                    except Exception as e:
                        print(f"  [WARN] Failed to load {fname}: {e}")

if loaded_nvidia_libs:
    print(f"[PASS] Successfully resolved and loaded {len(loaded_nvidia_libs)} NVIDIA shared library modules:")
    for lib in loaded_nvidia_libs:
        print(f"  + {lib}")
else:
    ld_path = os.environ.get("LD_LIBRARY_PATH", "")
    if any("nvidia" in p for p in ld_path.split(":")):
        print("[PASS] NVIDIA libraries configured via system LD_LIBRARY_PATH.")
    else:
        print("[INFO] No pip-installed NVIDIA libraries found; system CUDA/cuDNN will be used.")


1. ENVIRONMENT, PATHS & NVIDIA LIBRARY PRE-FLIGHT SETUP
Python Executable   : /home/ferret/src/terrain-diffusion/envs/conda/py_3.11/bin/python
Working Directory   : /home/ferret/src/terrain-diffusion/terrain_diffusion
Project Root        : /home/ferret/src/terrain-diffusion
CONDA_PREFIX        : /home/ferret/anaconda3
LD_LIBRARY_PATH     : :/home/ferret/.cache/JetBrains/PyCharm2026.2/acp-agents/junie/2783.5.0/junie-app/lib/app
CUDA_VISIBLE_DEVICES: (not set)

--- Scanning & Loading NVIDIA Shared Libraries from Site-Packages ---
[PASS] Successfully resolved and loaded 8 NVIDIA shared library modules:
  + libcudnn.so.9
  + libcudnn_adv.so.9
  + libcudnn_cnn.so.9
  + libcudnn_engines_precompiled.so.9
  + libcudnn_engines_runtime_compiled.so.9
  + libcudnn_graph.so.9
  + libcudnn_heuristic.so.9
  + libcudnn_ops.so.9


In [2]:
# Cell 2: PyTorch Installation, Hardware & CUDA Inspection
print("=" * 65)
print("2. PYTORCH INSTALLATION & HARDWARE INSPECTION")
print("=" * 65)

# 1. PyTorch import
try:
    import torch
    print(f"[PASS] PyTorch version        : {torch.__version__}")
except ImportError as e:
    print(f"[FAIL] PyTorch could not be imported: {e}")
    raise

# 2. CUDA support in PyTorch
torch_cuda_version = torch.version.cuda
cuda_available = torch.cuda.is_available()
print(f"PyTorch Built with CUDA       : {torch_cuda_version if torch_cuda_version else 'None (CPU build)'}")
print(f"CUDA Runtime Available        : {cuda_available}")

if cuda_available:
    device_count = torch.cuda.device_count()
    current_device = torch.cuda.current_device()
    device_name = torch.cuda.get_device_name(current_device)
    device_props = torch.cuda.get_device_properties(current_device)
    vram_gb = device_props.total_memory / (1024 ** 3)
    print(f"GPU Device Count              : {device_count}")
    print(f"Current Device Index          : {current_device}")
    print(f"GPU Device Name               : {device_name}")
    print(f"Compute Capability            : {device_props.major}.{device_props.minor}")
    print(f"Total Dedicated VRAM          : {vram_gb:.2f} GB")
else:
    print("[WARNING] CUDA runtime is not available. GPU acceleration will not be used.")

# 3. cuDNN configuration inspection
cudnn_avail = torch.backends.cudnn.is_available()
cudnn_enabled = torch.backends.cudnn.enabled
print(f"cuDNN Available               : {cudnn_avail}")
print(f"cuDNN Enabled                 : {cudnn_enabled}")
if cudnn_avail:
    try:
        cudnn_ver = torch.backends.cudnn.version()
        print(f"cuDNN Version                 : {cudnn_ver}")
    except Exception as e:
        print(f"[WARNING] Could not retrieve cuDNN version: {e}")


2. PYTORCH INSTALLATION & HARDWARE INSPECTION


[PASS] PyTorch version        : 2.13.0+cu130
PyTorch Built with CUDA       : 13.0
CUDA Runtime Available        : True
GPU Device Count              : 1
Current Device Index          : 0
GPU Device Name               : NVIDIA GeForce RTX 4070 SUPER
Compute Capability            : 8.9
Total Dedicated VRAM          : 11.59 GB
cuDNN Available               : True
cuDNN Enabled                 : True
cuDNN Version                 : 92000


In [3]:
# Cell 3: Functional PyTorch Execution Tests & CUDA Conv2d Validation
import traceback

print("=" * 65)
print("3. FUNCTIONAL PYTORCH & CUDA EXECUTION TESTS")
print("=" * 65)

test_results = {}

# Test 1: CPU Tensor Operations
try:
    x_cpu = torch.randn(100, 100)
    y_cpu = torch.matmul(x_cpu, x_cpu)
    assert y_cpu.shape == (100, 100)
    print("[PASS] Test 1: CPU Tensor Allocation and Matrix Multiplication")
    test_results["cpu_ops"] = "PASS"
except Exception as e:
    print(f"[FAIL] Test 1: CPU Tensor Operations failed: {e}")
    test_results["cpu_ops"] = f"FAIL: {e}"

# Test 2: CUDA Allocation & Matrix Multiplication
if torch.cuda.is_available():
    try:
        x_cuda = torch.randn(512, 512, device="cuda")
        y_cuda = torch.matmul(x_cuda, x_cuda)
        torch.cuda.synchronize()
        assert y_cuda.shape == (512, 512)
        allocated_mb = torch.cuda.memory_allocated() / (1024 ** 2)
        print(f"[PASS] Test 2: CUDA Tensor Allocation and GEMM (VRAM allocated: {allocated_mb:.2f} MB)")
        test_results["cuda_gemm"] = "PASS"
    except Exception as e:
        print(f"[FAIL] Test 2: CUDA GEMM failed: {e}")
        test_results["cuda_gemm"] = f"FAIL: {e}"
else:
    print("[SKIP] Test 2: CUDA GEMM (CUDA not available)")
    test_results["cuda_gemm"] = "SKIPPED (no CUDA)"

# Test 3: 2D Convolution on CUDA with cuDNN (Diffusion Core Operation)
if torch.cuda.is_available():
    try:
        conv = torch.nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1).to("cuda")
        inp = torch.randn(1, 16, 64, 64, device="cuda")
        out = conv(inp)
        torch.cuda.synchronize()
        assert out.shape == (1, 32, 64, 64)
        print(f"[PASS] Test 3: 2D Convolution on CUDA (cuDNN enabled={torch.backends.cudnn.enabled})")
        test_results["cuda_conv2d"] = "PASS"
    except Exception as e:
        print(f"[FAIL] Test 3: 2D Convolution on CUDA failed: {e}")
        test_results["cuda_conv2d"] = f"FAIL: {e}"
        print()
        print("--- Diagnostic Details for Conv2d Failure ---")
        traceback.print_exc()
        print()
        print("--- Diagnostic Recommendations ---")
        print("1. If the error is 'Invalid handle' or symbol resolution (e.g. cudnnGetVersion):")
        print("   Configure LD_LIBRARY_PATH or ensure nvidia/cudnn/lib is pre-loaded before calling Conv2d.")
        print("2. Alternatively, disable cuDNN to use PyTorch native CUDA kernels:")
        print("   torch.backends.cudnn.enabled = False")
else:
    print("[SKIP] Test 3: 2D Convolution on CUDA (CUDA not available)")
    test_results["cuda_conv2d"] = "SKIPPED (no CUDA)"

# Test 4: cuDNN Fallback Behavior
if torch.cuda.is_available():
    try:
        orig_state = torch.backends.cudnn.enabled
        torch.backends.cudnn.enabled = False
        conv_fb = torch.nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1).to("cuda")
        inp_fb = torch.randn(1, 16, 64, 64, device="cuda")
        out_fb = conv_fb(inp_fb)
        torch.cuda.synchronize()
        assert out_fb.shape == (1, 32, 64, 64)
        torch.backends.cudnn.enabled = orig_state
        print("[PASS] Test 4: Native CUDA Conv2d fallback (with cuDNN disabled)")
        test_results["cudnn_fallback"] = "PASS"
    except Exception as e:
        print(f"[FAIL] Test 4: cuDNN fallback failed: {e}")
        test_results["cudnn_fallback"] = f"FAIL: {e}"


3. FUNCTIONAL PYTORCH & CUDA EXECUTION TESTS
[PASS] Test 1: CPU Tensor Allocation and Matrix Multiplication
[PASS] Test 2: CUDA Tensor Allocation and GEMM (VRAM allocated: 10.12 MB)
[PASS] Test 3: 2D Convolution on CUDA (cuDNN enabled=True)
[PASS] Test 4: Native CUDA Conv2d fallback (with cuDNN disabled)


In [4]:
# Cell 4: Terrain Diffusion Integration & Repository Import Check
print("=" * 65)
print("4. TERRAIN DIFFUSION REPOSITORY IMPORT & PIPELINE CHECK")
print("=" * 65)

try:
    import terrain_diffusion
    from terrain_diffusion.paths import get_data_path, get_checkpoint_path, PROJECT_ROOT
    from terrain_diffusion.inference.tiff_export import export_tiff
    print(f"[PASS] terrain_diffusion imported successfully from: {terrain_diffusion.__file__}")
    print(f"Project root resolved to: {PROJECT_ROOT}")
    test_results["terrain_diffusion_import"] = "PASS"
except ImportError as e:
    print(f"[FAIL] Failed to import terrain_diffusion: {e}")
    test_results["terrain_diffusion_import"] = f"FAIL: {e}"
    print("Diagnosis: Ensure sys.path includes the project root directory containing terrain_diffusion.")
except Exception as e:
    print(f"[FAIL] Unexpected error importing terrain_diffusion: {e}")
    test_results["terrain_diffusion_import"] = f"FAIL: {e}"


4. TERRAIN DIFFUSION REPOSITORY IMPORT & PIPELINE CHECK


/home/ferret/src/terrain-diffusion/envs/conda/py_3.11/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


[PASS] terrain_diffusion imported successfully from: /home/ferret/src/terrain-diffusion/terrain_diffusion/__init__.py
Project root resolved to: /home/ferret/src/terrain-diffusion


In [5]:
# Cell 5: PyTorch Health & Diagnostic Summary Report
print("=" * 65)
print("5. PYTORCH CONFIGURATION & HEALTH SUMMARY REPORT")
print("=" * 65)

all_passed = True
for test_name, status in test_results.items():
    indicator = "[PASS]" if status == "PASS" else ("[SKIP]" if "SKIPPED" in status else "[FAIL]")
    if "FAIL" in status:
        all_passed = False
    print(f"{indicator} {test_name:<28} : {status}")

print("-" * 65)
if all_passed:
    print("STATUS: PyTorch configuration is fully verified, functional, and ready for Terrain Diffusion.")
else:
    print("STATUS: Issues detected during PyTorch testing.")
    print()
    print("Actionable Diagnostics & Troubleshooting Steps:")
    if test_results.get("cuda_conv2d", "").startswith("FAIL"):
        print(" - cuDNN Conv2d failed. If the error mentions symbol resolution (cudnnGetVersion),")
        print("   ensure nvidia/cudnn/lib is added to LD_LIBRARY_PATH or loaded via ctypes.")
        print("   You can also set torch.backends.cudnn.enabled = False to use native CUDA kernels.")
    if test_results.get("cuda_gemm", "").startswith("FAIL"):
        print(" - CUDA GEMM failed. Verify NVIDIA GPU driver compatibility and PyTorch CUDA build.")
    if test_results.get("terrain_diffusion_import", "").startswith("FAIL"):
        print(" - Module import failed. Check that the project root is added to sys.path.")


5. PYTORCH CONFIGURATION & HEALTH SUMMARY REPORT
[PASS] cpu_ops                      : PASS
[PASS] cuda_gemm                    : PASS
[PASS] cuda_conv2d                  : PASS
[PASS] cudnn_fallback               : PASS
[PASS] terrain_diffusion_import     : PASS
-----------------------------------------------------------------
STATUS: PyTorch configuration is fully verified, functional, and ready for Terrain Diffusion.
